In [0]:
df = spark.read.option("multiLine", "true").json("abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/fred_raw/data/DCOILWTICO/")
df.printSchema()
df.show(1, truncate=50)

In [0]:
import json
import io
import zipfile
import datetime as dt
import requests
from typing import Dict, Any, Optional

# ============================================================================
# StatCan Ingest — Supply-Side Data for Canada Electronics Lakehouse
# ============================================================================
# Industry techniques demonstrated:
#   1. Multi-step API workflow (JSON metadata call → ZIP download → CSV extract)
#   2. Handling messy government CSV (14+ columns, status codes, scalar factors)
#   3. Writing raw CSV to bronze (preserving original messiness for Silver to clean)
#   4. Manifest pattern (consistent with FRED / WITS ingests)
# ============================================================================

# ---------------- Config ----------------
CATALOG = "canada_business"
SCHEMA  = "bronze"
BRONZE_SUBDIR = "statcan_raw"

# StatCan Product IDs (PIDs) — these are the table numbers
TABLES: Dict[str, str] = {
    "20100056": "Monthly retail trade sales by NAICS (current)",
    "20100076": "Retail inventory to sales ratio (monthly)",
    "12100121": "International merchandise trade by HS chapter",
}

STATCAN_BASE = "https://www150.statcan.gc.ca/t1/wds/rest"

# ---------------- Helpers ----------------
def run_ts_utc() -> str:
    return dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def resolve_schema_root_location(catalog: str, schema: str) -> str:
    df = spark.sql(f"DESCRIBE SCHEMA EXTENDED {catalog}.{schema}").toPandas()
    rows = df.loc[df["database_description_item"] == "RootLocation", "database_description_value"].values
    if len(rows) == 0 or not rows[0]:
        raise RuntimeError(f"RootLocation not found for {catalog}.{schema}")
    return str(rows[0])

def join_uri(base_uri: str, child: str) -> str:
    return base_uri.rstrip("/") + "/" + child.strip("/")

def get_csv_download_url(pid: str) -> str:
    """
    TECHNIQUE: Two-step StatCan API workflow
    Step 1: Call getFullTableDownloadCSV/{PID}/en → returns JSON with a URL
    Step 2: Download the ZIP from that URL
    This indirection is how StatCan serves large files — the API generates
    a temporary download link rather than serving the file directly.
    """
    url = f"{STATCAN_BASE}/getFullTableDownloadCSV/{pid}/en"
    headers = {"User-Agent": "databricks-lakehouse-demo/1.0"}
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()
    payload = r.json()

    # Response format: {"status": "SUCCESS", "object": "https://...zip"}
    if payload.get("status") != "SUCCESS":
        raise RuntimeError(f"StatCan API error for PID {pid}: {payload}")

    return payload["object"]

def download_and_extract_csv(zip_url: str, pid: str) -> Dict[str, str]:
    """
    TECHNIQUE: In-memory ZIP extraction
    StatCan delivers full table downloads as ZIP files containing:
      - {PID}.csv (the data file — this is what we want)
      - {PID}_MetaData.csv (metadata file — column descriptions, notes)
    We extract both and return as {filename: content_string} dict.

    The data CSV is intentionally messy:
      - 14+ columns with verbose names
      - Bilingual column names possible
      - STATUS column with symbol codes (F, x, .., etc.)
      - SCALAR_FACTOR and UOM columns that vary by row
      - Multi-level hierarchy codes in dimension columns
    This messiness is preserved in Bronze — Silver will clean it.
    """
    headers = {"User-Agent": "databricks-lakehouse-demo/1.0"}
    r = requests.get(zip_url, headers=headers, timeout=300)
    r.raise_for_status()

    files = {}
    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        for name in zf.namelist():
            if name.endswith(".csv"):
                files[name] = zf.read(name).decode("utf-8-sig")  # BOM handling
    return files

# ---------------- Resolve UC paths ----------------
root_location = resolve_schema_root_location(CATALOG, SCHEMA)
bronze_base = join_uri(root_location, BRONZE_SUBDIR)
data_base = bronze_base.rstrip("/") + "/data"
manifest_base = bronze_base.rstrip("/") + "/manifests"

dbutils.fs.mkdirs(data_base)
dbutils.fs.mkdirs(manifest_base)

# ---------------- Ingest loop ----------------
ts = run_ts_utc()

manifest = {
    "run_ts": ts,
    "catalog": CATALOG,
    "schema": SCHEMA,
    "bronze_base": bronze_base,
    "source": "Statistics Canada Full Table Download (CSV)",
    "tables": []
}

for pid, description in TABLES.items():
    print(f"⏳ Fetching PID {pid}: {description}")

    # Step 1: Get download URL from StatCan API
    zip_url = get_csv_download_url(pid)
    print(f"  ZIP URL: {zip_url}")

    # Step 2: Download ZIP and extract CSVs
    csv_files = download_and_extract_csv(zip_url, pid)

    table_manifest = {
        "pid": pid,
        "description": description,
        "zip_url": zip_url,
        "files": []
    }

    # Step 3: Write each CSV to bronze storage
    for filename, content in csv_files.items():
        out_dir = join_uri(data_base, pid)
        dbutils.fs.mkdirs(out_dir)

        out_file = join_uri(out_dir, f"{filename.replace('.csv', '')}_{ts}.csv")
        dbutils.fs.put(out_file, content, overwrite=False)

        row_count = content.count("\n") - 1  # rough estimate (header row)
        print(f"  ✅ {filename} -> ~{row_count} rows -> {out_file}")

        table_manifest["files"].append({
            "original_filename": filename,
            "raw_file": out_file,
            "approx_rows": row_count,
            "bytes": len(content)
        })

    manifest["tables"].append(table_manifest)

# Step 4: Write manifest
manifest_path = join_uri(manifest_base, f"run_manifest_{ts}.json")
dbutils.fs.put(manifest_path, json.dumps(manifest, indent=2), overwrite=False)

print("\n✅ StatCan ingest complete")
print("Bronze base:", bronze_base)
print("Manifest:", manifest_path)
for t in manifest["tables"]:
    for f in t["files"]:
        print(f"  ✅ PID {t['pid']} | {f['original_filename']} | ~{f['approx_rows']} rows | {f['raw_file']}")